#### From week 3 section 

- `Early Data Analysis`
- `Duplicate Correction`
- `Inconsistencies Fixing`
- `Missing Data Checking`
- `Planning for EDA`

In [98]:
import pandas as pd
import plotly.express as px

years = list(range(2020, 2027))
dfs = []

for year in years:
    for quarter in ["Q1", "Q2", "Q3", "Q4"]:
        file_path = f"../data/raw/BDD PRODUCCION/BDD PRODUCCION/{year}/{quarter} {year}.csv"
        try:
            df = pd.read_csv(file_path)

            if year == 2025 and quarter == "Q2":
                s = df["order_date"]
                p1 = pd.to_datetime(s, dayfirst=True, errors="coerce")
                p2 = pd.to_datetime(s, dayfirst=False, errors="coerce")
                num = pd.to_numeric(s, errors="coerce")
                excel = pd.to_datetime(num, unit="D", origin="1899-12-30")
                df["order_date"] = p1.fillna(p2).fillna(excel)
            else:
                df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

            dfs.append(df)
        except FileNotFoundError as e:
            print(f"File not found: {file_path}, error: {e}")

remissions = pd.concat(dfs, ignore_index=True)
remissions.head()


File not found: ../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q2 2026.csv, error: [Errno 2] No such file or directory: '../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q2 2026.csv'
File not found: ../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q3 2026.csv, error: [Errno 2] No such file or directory: '../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q3 2026.csv'
File not found: ../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q4 2026.csv, error: [Errno 2] No such file or directory: '../data/raw/BDD PRODUCCION/BDD PRODUCCION/2026/Q4 2026.csv'


,Year,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
0,2020,51013232,2020-02-04,1073,2020-02-04 07:00:00,7044,510,6.0,2020-02-04 06:45:13,2020-02-04 08:17:24,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
1,2020,51013233,2020-02-04,1073,2020-02-04 07:00:00,6603,510,5.5,2020-02-04 06:45:22,2020-02-04 08:26:06,101,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
2,2020,51013235,2020-02-04,1028,2020-02-04 08:00:00,9631,510,2.5,2020-02-04 07:37:56,2020-02-04 08:55:49,78,ONE TIME ESP ANGELICA RIVERA GOMEZ,ROMERO JULIA,CALLE MANUEL BECERRA 14515 COL ALAMEDAS,CH-F5
3,2020,51013236,2020-02-04,1005,2020-02-04 08:15:00,6598,510,3.0,2020-02-04 08:02:36,2020-02-04 09:47:43,105,IVAN NOE SIMENTAL ORTEGA,COLECTOR SACRAMENTO,ARROLLO MIMBRE Y VIALIDAD SACRAMENTO,CH-J9
4,2020,51013238,2020-02-04,1026,2020-02-04 08:30:00,10145,510,3.5,2020-02-04 08:09:01,2020-02-04 09:09:15,60,ONE TIME CONSTRUCENTRO CHIH,MOLINA BALDERRAMA CESAR,PERIF DE LA JUVENTUD 9926 COL RESIDENCIA,CH-L5


NO MOVER ESTE CODIGO (Arregla issue con formato de Q2 2025.csv)

In [99]:
order_date = pd.to_datetime(remissions["order_date"], errors="coerce").dt.normalize()

def rebuild_datetime(col):
    raw = remissions[col].astype(str).str.strip()

    time_str = raw.str.extract(r"(\d{1,2}:\d{2}(?::\d{2})?\s*[APMapm]{0,2})")[0]
    time_str = time_str.str.replace(r"\.\d+", "", regex=True)

    t24 = pd.to_datetime(time_str, format="%H:%M:%S", errors="coerce")
    t24 = t24.fillna(pd.to_datetime(time_str, format="%H:%M", errors="coerce"))

    t12 = pd.to_datetime(time_str, format="%I:%M:%S %p", errors="coerce")
    t12 = t12.fillna(pd.to_datetime(time_str, format="%I:%M %p", errors="coerce"))

    t = t24.fillna(t12)
    time_only = t - t.dt.normalize()

    return order_date + time_only

remissions["typed_time"] = rebuild_datetime("typed_time")
remissions["start_time"] = rebuild_datetime("start_time")
remissions["at_plant_time"] = rebuild_datetime("at_plant_time")

In [100]:
remissions_df = pd.read_csv("../data/processed/remissions_db.csv")
remissions_df.head(-1)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
0,51013232,2/4/2020 0:00,1073,2/4/2020 7:00,7044,510,6.0,2/4/2020 6:45,2/4/2020 8:17,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
1,51013233,2/4/2020 0:00,1073,2/4/2020 7:00,6603,510,5.5,2/4/2020 6:45,2/4/2020 8:26,101,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
2,51013235,2/4/2020 0:00,1028,2/4/2020 8:00,9631,510,2.5,2/4/2020 7:37,2/4/2020 8:55,78,ONE TIME ESP ANGELICA RIVERA GOMEZ,ROMERO JULIA,CALLE MANUEL BECERRA 14515 COL ALAMEDAS,CH-F5
3,51013236,2/4/2020 0:00,1005,2/4/2020 8:15,6598,510,3.0,2/4/2020 8:02,2/4/2020 9:47,105,IVAN NOE SIMENTAL ORTEGA,COLECTOR SACRAMENTO,ARROLLO MIMBRE Y VIALIDAD SACRAMENTO,CH-J9
4,51013238,2/4/2020 0:00,1026,2/4/2020 8:30,10145,510,3.5,2/4/2020 8:09,2/4/2020 9:09,60,ONE TIME CONSTRUCENTRO CHIH,MOLINA BALDERRAMA CESAR,PERIF DE LA JUVENTUD 9926 COL RESIDENCIA,CH-L5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340768,71037451,1/31/2026 0:00,1069,1/31/2026 11:00,13053,710,2.0,1/31/2026 10:06,1/31/2026 11:03,57,BENJAMIN MATA CONDE,VILLAS LA HACIENDITA,EJIDO LA HACIENDITA S/N CHIHUAHUA,CHP-4
340769,71037452,1/31/2026 0:00,1073,1/31/2026 10:00,12831,710,3.0,1/31/2026 10:36,1/31/2026 10:50,14,FERRETERIA MOLINA DE CHIHUAHUA,FERRETERIA MOLINA DE CHIHUAHUA,FRACC DOMINIO LOTE 2 MANZANA 10,CHN-3
340770,71037453,1/31/2026 0:00,1244,1/31/2026 10:00,12831,710,0.5,1/31/2026 10:51,1/31/2026 13:01,130,FERRETERIA MOLINA DE CHIHUAHUA,FERRETERIA MOLINA DE CHIHUAHUA,FRACC DOMINIO LOTE 2 MANZANA 10,CHN-3
340771,71037454,1/31/2026 0:00,1070,1/31/2026 12:00,13041,710,2.5,1/31/2026 10:55,1/31/2026 12:52,117,ALFONSO ORTEGA ANTILLON,FRAC ALCAZARES (MYKONOS),AVE DE LA CANTERA SN ALCAZARES (MYKON,CHP-3


In [101]:
# Based on EDA analysis, it´s okay to drop rows with null values (<2%)
remissions_df = remissions_df[remissions_df['ship_addr_line'].notnull() & remissions_df['map_page'].notnull() & remissions_df['name'].notnull()]
print("Number of rows after dropping null values:", remissions_df.shape[0])

Number of rows after dropping null values: 340244


In [102]:
# Change 'order_date' and 'typed_time' to datetime format
remissions_df['order_date'] = pd.to_datetime(remissions_df['order_date'], errors='raise', format='mixed')
remissions_df['typed_time'] = pd.to_datetime(remissions_df['typed_time'], errors='raise', format='mixed')
remissions_df['start_time'] = pd.to_datetime(remissions_df['start_time'], errors='raise', format='mixed')
remissions_df['at_plant_time'] = pd.to_datetime(remissions_df['at_plant_time'], errors='raise', format='mixed')
remissions_df.info()

<class 'pandas.DataFrame'>
Index: 340244 entries, 0 to 340773
Data columns (total 14 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   tkt_code             340244 non-null  int64         
 1   order_date           340244 non-null  datetime64[us]
 2   order_code           340244 non-null  int64         
 3   start_time           340244 non-null  datetime64[us]
 4   truck_code           340244 non-null  int64         
 5   ship_plant_code      340244 non-null  int64         
 6   u_Volumen            340244 non-null  float64       
 7   typed_time           340244 non-null  datetime64[us]
 8   at_plant_time        340244 non-null  datetime64[us]
 9   u_Cicle              340244 non-null  int64         
 10  name                 340244 non-null  str           
 11  Nombre del proyecto  340244 non-null  str           
 12  ship_addr_line       340244 non-null  str           
 13  map_page             340244 no

In [103]:
# Duplicate check for all time columns
typed_time_duplicates = remissions_df[remissions_df.duplicated(subset=['start_time', 'at_plant_time','truck_code'], keep=False)]
typed_time_duplicates = typed_time_duplicates.sort_values(by='typed_time', ascending=True)
typed_time_duplicates.head(10)

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
29740,51261174,2020-09-08,1201,2020-09-08 10:30:00,7039,512,1.5,2020-09-08 09:29:00,2020-09-08 10:47:00,78,RUBA DESARROLLOS,MOLINA FRACC VENDANOVA 58 VIV,MOLINA FRACC VENDANOVA 58 VIV S/N MOL,CH-C4
29741,51261175,2020-09-08,1239,2020-09-08 10:30:00,7039,512,1.0,2020-09-08 09:30:00,2020-09-08 10:47:00,77,RUBA DESARROLLOS,ESTRADA VENDANOVA 42VIV,ESTRADA VENDANOVA 42VIV S/N ESTRADA V,CH-B1
77566,51038180,2021-08-12,1155,2021-08-12 10:00:00,6163,510,5.0,2021-08-12 10:22:00,2021-08-12 11:54:00,92,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77567,51038180,2021-08-30,1243,2021-08-12 10:00:00,6163,510,5.0,2021-08-12 10:22:00,2021-08-12 11:54:00,92,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77575,51038191,2021-08-12,1155,2021-08-12 10:00:00,9430,510,5.0,2021-08-12 11:39:00,2021-08-12 12:32:00,53,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77576,51038191,2021-08-30,1243,2021-08-12 10:00:00,9430,510,5.0,2021-08-12 11:39:00,2021-08-12 12:32:00,53,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77579,51038194,2021-08-12,1155,2021-08-12 10:00:00,9431,510,5.5,2021-08-12 12:00:00,2021-08-12 13:22:00,82,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77580,51038194,2021-08-30,1243,2021-08-12 10:00:00,9431,510,5.5,2021-08-12 12:00:00,2021-08-12 13:22:00,82,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77582,51038199,2021-08-12,1155,2021-08-12 10:00:00,10145,510,5.5,2021-08-12 12:20:00,2021-08-12 14:02:00,102,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77583,51038199,2021-08-30,1243,2021-08-12 10:00:00,10145,510,5.5,2021-08-12 12:20:00,2021-08-12 14:02:00,102,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1


In [104]:
print("Number of duplicate rows based on start_time, at_plant_time, and truck_code:", typed_time_duplicates.shape[0])

Number of duplicate rows based on start_time, at_plant_time, and truck_code: 32


In [105]:
# It is impossible for many trucks to have the same start_time and at_plant_time
# Therefore, these are duplicates that should be removed, as they are likely to be errors in the data entry process.
remissions_df = remissions_df.drop(typed_time_duplicates.index)
print("Number of rows after dropping duplicates:", remissions_df.shape[0])

Number of rows after dropping duplicates: 340212


We will check u_Cicle to see if there is any order that was never completed.

In [106]:
# Graph the distribution of u_Cicle to see if there are any outliers
fig = px.histogram(remissions_df, x='u_Cicle', nbins=len(remissions_df['u_Cicle'].unique()), title='Distribution of u_Cicle')
fig.show()

In [107]:
# Boxplot of u_Cicle to check for outliers
fig = px.box(remissions_df, y='u_Cicle', title='Boxplot of u_Cicle')
fig.show()

We will check distribution of volume amount

In [108]:
# Calculate counts and percentages for u_Volumen
counts = remissions_df['u_Volumen'].value_counts().sort_index()
percentages = (counts / counts.sum()) * 100

# Histogram of frequency of each unique volume value to check for outliers
x_labels = counts.index.astype(str)  # treat volumes as categorical so bars are wider
fig = px.bar(x=x_labels, y=counts.values, title='Distribution of Volume',
             labels={'x':'u_Volumen','y':'count'}, text=counts.values)
fig.update_traces(
    textposition='outside',
    texttemplate='%{y}',
    customdata=percentages.values,
    hovertemplate='<b>Volume:</b> %{x}<br><b>Count:</b> %{y}<br><b>Percentage:</b> %{customdata:.2f}%<extra></extra>'
)
fig.update_layout(
    bargap=0.1,        # reduce gap between bars
    xaxis_tickangle=45,
    width=1000
)
fig.show()

In [109]:
# Sum all the tkt_code count and u_Volumen related to each order_code and print the orders with highest to lowest total volume and remissions
order_volume = remissions_df.groupby('order_code').agg(
    u_Volumen=('u_Volumen', 'sum'),
    tkt_code_sum=('tkt_code', 'count')
).reset_index()

order_volume = order_volume.sort_values(by='u_Volumen', ascending=False)
print(order_volume.head(10))

     order_code  u_Volumen  tkt_code_sum
0          1000    4753.50          1024
113        1113    4697.00          1279
86         1086    4647.50          1197
70         1070    4532.50          1154
107        1107    4333.50          1159
54         1054    4229.75          1074
7          1007    4170.50           978
128        1128    4039.00          1142
49         1049    4038.50          1051
101        1101    4024.50          1099


In [110]:
# print the count of different map_page values associated with the order_code = 1000
order_1000_map_pages = remissions_df[remissions_df['order_code'] == 1000]['map_page'].nunique()
print("Count of different map_page values associated with order_code 1000:", order_1000_map_pages)

Count of different map_page values associated with order_code 1000: 116


In [111]:
# print the oldest date of the tkt_code and the newest of the order_code = 1000
order_1000_dates = remissions_df[remissions_df['order_code'] == 1000]['order_date']
oldest_date = order_1000_dates.min()
newest_date = order_1000_dates.max()
print("Oldest date of tkt_code for order_code 1000:", oldest_date)
print("Newest date of tkt_code for order_code 1000:", newest_date)

Oldest date of tkt_code for order_code 1000: 2020-02-05 00:00:00
Newest date of tkt_code for order_code 1000: 2026-01-29 00:00:00


In [112]:
# print the count of how many unique order_code are in total
unique_order_codes = remissions_df['order_code'].nunique()
print("Count of unique order_code in total:", unique_order_codes)

Count of unique order_code in total: 778


In [113]:
# Distribution with x values being the order_code unique values and y values being the count of tkt_code for each order_code
order_code_counts = remissions_df['order_code'].value_counts().sort_index()

fig = px.bar(
    x=order_code_counts.index.astype(str),
    y=order_code_counts.values,
    title='Distribution of tkt_code count by order_code',
    labels={'x': 'order_code', 'y': 'tkt_code count'},
    text=order_code_counts.values
)
fig.update_traces(
    textposition='outside',
    texttemplate='%{y}',
    hovertemplate='<b>Order Code:</b> %{x}<br><b>tkt_code Count:</b> %{y}<extra></extra>'
)
fig.update_layout(
    bargap=0.1,
    xaxis_showticklabels=False,
    width=1000
)
fig.show()


In [114]:
# make a boxplot of the previous graph to check for outliers
fig = px.box(order_code_counts.values, title='Boxplot of tkt_code count by order_code')
fig.update_layout(width=800)
fig.show()

In [115]:
# Check distribution of plants where u_Cicle is 1
u_cicle_1 = remissions_df[remissions_df['u_Cicle'] == 1]
plant_counts = u_cicle_1['ship_plant_code'].value_counts()
print(plant_counts)

ship_plant_code
512    4070
515    1651
710    1363
514    1338
511     748
510     316
717     203
Name: count, dtype: int64


In [116]:
#Looks like the tkt_codes can be repeated after years of orders.
remissions_df[remissions_df['tkt_code'].duplicated(keep=False)].sort_values('tkt_code', ascending=True).head(6)


,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
1,51013233,2020-02-04,1073,2020-02-04 07:00:00,6603,510,5.5,2020-02-04 06:45:00,2020-02-04 08:26:00,101,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
310446,51013233,2025-08-02,1035,2025-08-02 07:00:00,9430,510,2.0,2025-08-02 06:46:00,2025-08-02 07:46:00,60,ONE TIME PROMOCIONES OCTAVIO RIOS,MARGARITO ROMERO,ING. CARRILLO 16344 COL TRAHUMARA,CHJ-3
310448,51013236,2025-08-02,1201,2025-08-02 07:00:00,7043,510,4.0,2025-08-02 06:54:00,2025-08-02 08:14:00,80,JONATHAN MARQUEZ BACA,OBRAS VARIAS,ARROYO NARAGUA #2225 LOS ARROYOS,CHD-5
3,51013236,2020-02-04,1005,2020-02-04 08:15:00,6598,510,3.0,2020-02-04 08:02:00,2020-02-04 09:47:00,105,IVAN NOE SIMENTAL ORTEGA,COLECTOR SACRAMENTO,ARROLLO MIMBRE Y VIALIDAD SACRAMENTO,CH-J9
5,51013239,2020-02-04,1150,2020-02-04 09:00:00,6611,510,6.0,2020-02-04 08:36:00,2020-02-04 09:30:00,54,JULIO ARMANDO HINOJOS ENRIQUEZ,FRAC CALZADA DEL BOSQUE AGH,FRAC CALZADA DEL BOSQUE AGH FRAC CAL,CH-H3
310450,51013239,2025-08-02,1090,2025-08-02 08:00:00,9430,510,6.5,2025-08-02 07:46:00,2025-08-02 09:03:00,77,FERRETERIA MOLINA DE CHIHUAHUA,FERRETERIA MOLINA DE CHIHUAHUA,MONTE HIMALAYA 4341 QUINTAS CAROLINA,CHH-8


We will now check the consistency of remissions per plant

In [117]:
print(remissions_df['ship_plant_code'].unique())
# The new plant (717) was not used for the past version of this project

[510 511 512 515 710 717 514]


In [118]:
counts = remissions_df['ship_plant_code'].value_counts()
percentages = (counts / counts.sum()) * 100
print(counts, '\t', percentages.round(2)) # percentages
print("total:", counts.sum())

ship_plant_code
512    79483
510    68930
511    58374
515    49380
710    44724
514    37317
717     2004
Name: count, dtype: int64 	 ship_plant_code
512    23.36
510    20.26
511    17.16
515    14.51
710    13.15
514    10.97
717     0.59
Name: count, dtype: float64
total: 340212


In [119]:
# See lowest and highest datetime for each plant
for plant_code in remissions_df['ship_plant_code'].unique():
    plant = remissions_df[remissions_df['ship_plant_code'] == plant_code]
    print(f"Plant {plant_code}:")
    print(f"  Lowest datetime: {plant['start_time'].min()}")
    print(f"  Highest datetime: {plant['start_time'].max()}")

Plant 510:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-05-28 22:00:00
Plant 511:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-05-28 22:00:00
Plant 512:
  Lowest datetime: 2020-02-04 05:00:00
  Highest datetime: 2026-05-28 19:00:00
Plant 515:
  Lowest datetime: 2020-02-04 08:30:00
  Highest datetime: 2026-05-28 22:00:00
Plant 710:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-05-28 17:45:00
Plant 717:
  Lowest datetime: 2020-02-04 07:30:00
  Highest datetime: 2022-02-09 07:00:00
Plant 514:
  Lowest datetime: 2022-07-20 07:00:00
  Highest datetime: 2026-05-28 22:00:00


In [120]:
# Drop plant 717 due to low number of remissions and being inactive for the past 4 years
remissions_df = remissions_df[remissions_df['ship_plant_code'] != 717]
print("Number of rows after dropping plant 717:", remissions_df.shape[0])

# Also plant 514 starts from 2022, but that is no problem as it covers over a 10% of the total remission count

Number of rows after dropping plant 717: 338208


In [121]:
# There are volume outliers based on EDA, so we will find them and drop them
# print the rows in which `u_Volumen` is greater than 7 or less/equal than 0
outliers_volumen = remissions_df[(remissions_df['u_Volumen'] > 7) | (remissions_df['u_Volumen'] <= 0)]
print("Rows with volume outliers:")
print(outliers_volumen.shape[0])

Rows with volume outliers:
0


In order to identify repeated orders, the row would need to have repeated values in the following columns: order_code, order_date, typed_time, truck_code. 

In [122]:
# Look for repeated order_codes

repeated_orders = remissions_df[remissions_df.duplicated(subset=['order_code', 'order_date','typed_time','truck_code'], keep=False)]
repeated_orders.head(-1)

# No repeated orders.

,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page


In [123]:
# Count unique values for each column
unique_counts = remissions_df.nunique()
print(unique_counts)

tkt_code               323948
order_date               1688
order_code                777
start_time              64991
truck_code                149
ship_plant_code             6
u_Volumen                  32
typed_time             281600
at_plant_time          267642
u_Cicle                   209
name                     2019
Nombre del proyecto     22844
ship_addr_line          70753
map_page                  952
dtype: int64


In [124]:
remissions_df.shape[0]

338208

In [125]:
# print unique values per column



Based on the initial unique values for each column:
- There have been 777 orders in total
- There have been 149 trucks
- There have been 22844 different projects
- There have been 2019 different clients (apparently)
- There have been 70753 different addresses for delivery

#### Imputation Section

In [126]:
# 1) Parse dates (coerce invalid to NaT so we can measure failures)
remissions_df['order_date_parsed'] = pd.to_datetime(
    remissions_df['order_date'], errors='coerce'
)

# 2) % of rows that failed to parse
remissions_df['order_date_parsed'].isna().mean()

np.float64(0.0)

#### Exporting Cleaned Dataset

In [127]:
# Has to be xlsx because of datetime format
remissions_df.to_excel("../data/processed/remissions_db_cleaned.xlsx", index=False)